# 전기차 에너지 소비량 예측 모델 구축
### 주행 조건 기반 전비(kWh/100km) 예측 — OEM 내비게이션 에너지 비용 함수 개발

---

## 0. 문제 정의

### 업무 이슈
전기차의 실제 전비는 주행 조건에 따라 **11.62 ~ 35.00 kWh/100km** 로 나타난다.
60kWh 배터리 기준 주행가능거리로 환산하면 약 **516km ~ 171km**, 조건별 스프레드가 **약 345km**다.
그러나 현재 내비게이션은 카탈로그 공인 전비(고정값) 하나로 경로·잔량을 계산하므로,
운전자는 주행거리 불안과 불필요한 충전 대기를 겪는다.

> 위 345km는 *조건에 따라 가능한 주행거리의 범위*이지 특정 방식의 오차가 아니다.
> 고정 전비 방식이 실제로 얼마나 틀리는지는 5장에서 기준선 모델을 세워 측정한다.

### 문제 재정의 (업무 이슈 → 예측 대상 → 분석 목표 → 활용 방안)

| 단계 | 내용 |
|---|---|
| 업무 이슈 | 고정 전비 기반 경로 안내 → 도착 잔량 예측 불신 |
| 예측 대상 | 주행 조건별 **전비 (kWh/100km)** |
| 분석 목표 | 양산차 기존 신호만으로 전비를 예측하는 회귀 모델 구축·비교·선정 |
| 활용 방안 | 링크별 에너지 비용 함수 → 최소 에너지 경로·도착 SOC 안내 |

### 설계상의 핵심 정의 3가지

**① 분석 단위 = 균질 주행 조건 단위(링크).** 본 데이터의 1행은 경사·속도·부하가 일정하다고 간주되는
주행 단위다. 실제 경로는 내비게이션이 이미 링크 단위로 분해해 다루므로, 경로 에너지는 링크 예측값의 합으로 계산한다.

**② 타깃 = 총 소비량(kWh)이 아닌 전비(kWh/100km).** 총 소비량을 직접 예측하면 거리가 지배 변수가 되어
모델이 사실상 거리 곱셈기로 퇴화한다. 거리의 영향을 구조적으로 분리하기 위해 강도(intensity) 지표인 전비를
예측 대상으로 정의한다.

**③ 거리는 모델 입력이 아니라 변환 계수.** 예측 전비를 총에너지로 바꾸는 결정론적 계산에만 사용한다.
그 근거는 3-2에서 분할 불변성 요건으로 정식화한다.

> **문제 유형 선언** : 본 문제는 출발 시점에 사전 관측 가능한 주행·차량 조건을 활용하여
> 연속형 전비를 예측하는 **지도학습 기반 회귀 문제**로 정의한다.

### 모델 선정 기준
차량 탑재형 내비게이션이라는 활용처의 제약에 따라, 최종 모델은
**예측 정확도 · 추론 속도 · 해석 가능성 · 서비스 일관성** 4축으로 종합 평가하여 선정한다.


In [ ]:
# 환경 설정
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     learning_curve, RepeatedKFold, cross_val_score)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib, sklearn

# 한글 폰트 (환경에 있는 첫 번째 후보 사용)
import matplotlib.font_manager as fm
for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic']:
    if any(font.name == f for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = f
        break
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
np.random.seed(SEED)
print('scikit-learn', sklearn.__version__)

---
## 1. 데이터 이해

### 1-1. 데이터 구조 확인
8,000건의 주행 기록. **1행 = 1개의 균질 주행 단위(링크)** — 입력 조건과 그 결과 전비 1개.

In [ ]:
df = pd.read_csv('ev_energy_consumption.csv')
print('shape:', df.shape)
df.head()

In [ ]:
df.info()

### 1-2. 변수 ↔ 차량 신호 매핑 (사전 관측 가능성 점검)

모델이 실제 서비스에서 작동하려면 **모든 입력이 예측 시점(경로 탐색 시점)에 관측 가능**해야 한다.
이것이 개념 수준의 데이터 누수(leakage) 점검이다.

| 변수 | 의미 | 신호 출처 | 예측 시점 관측 | 역할 |
|---|---|---|---|---|
| `speed_kmh` | 링크 평균속도 | 실시간 교통정보 (경로별 예상속도) | ✅ | 모델 입력 |
| `road_grade_pct` | 도로 경사 | HD맵 고도 데이터 | ✅ | 모델 입력 |
| `payload_kg` | 적재중량 | 하중 센서 / 사용자 입력 | ✅ | 모델 입력 |
| `ambient_temp_C` | 외기온도 | 외기온 센서 | ✅ | 모델 입력 |
| `hvac_power_kw` | 공조 소비전력 | 공조 제어기 (설정 기준 부하) | ✅ | 모델 입력 |
| `battery_temp_C` | 배터리 온도 | BMS | ✅ | 모델 입력 |
| `tire_pressure_bar` | 타이어 공기압 | TPMS | ✅ | 모델 입력 |
| `driving_style_index` | 운전 성향 (0~1) | 차량 축적 운전자 프로파일 | ✅ | 모델 입력 |
| `trip_distance_km` | 링크 거리 | 경로 탐색 결과 | ✅ | **변환 계수** (3-2 참조) |

→ 전 변수가 양산차 기존 신호로 사전 관측 가능하다. 추가 하드웨어 없이 임베디드 탑재가 가능하다는
기획 전제가 데이터 수준에서 성립한다. 단, 거리는 예측 입력이 아닌 변환 계수로 쓴다(3-2).

### 1-3. 데이터 품질 점검

In [ ]:
print('결측치 합계:', df.isna().sum().sum())
print('중복 행:', df.duplicated().sum())

checks = {
    'speed_kmh >= 0': (df['speed_kmh'] >= 0).all(),
    'payload_kg >= 0': (df['payload_kg'] >= 0).all(),
    'hvac_power_kw >= 0': (df['hvac_power_kw'] >= 0).all(),
    'tire_pressure_bar > 0': (df['tire_pressure_bar'] > 0).all(),
    'trip_distance_km > 0': (df['trip_distance_km'] > 0).all(),
    'energy_consumption > 0': (df['energy_consumption_kwhper100km'] > 0).all(),
}
for k, v in checks.items():
    print(f'{k}: {"통과" if v else "위반"}')

df.describe().round(2).T

**판단** : 결측 0건, 중복 0건, 물리 범위 위반 0건. 별도의 결측 대체·이상치 제거 없이 전량(8,000건)을 사용한다.

---
## 2. 탐색적 데이터 분석 (EDA)

시각화 나열이 아니라 **각 분석이 어떤 판단으로 이어졌는지**를 기록한다.
분석 순서: 타깃 → 입력 분포 → 변수 간 구조(히트맵·VIF) → 입력-타깃 관계(형태) → 핵심 패턴 상세.

이 단계에서는 `trip_distance_km`도 다른 변수와 동일하게 분석한다.
모델 입력에서 제외하는 결정은 EDA 결과를 근거로 3-2에서 내린다.

### 2-1. 타깃 분포 — 변환이 필요한가?

In [ ]:
from scipy import stats

t = df['energy_consumption_kwhper100km']
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(t, bins=40, density=True, color='#5B7C99', alpha=0.85, edgecolor='white')
x = np.linspace(t.min(), t.max(), 200)
ax.plot(x, stats.norm.pdf(x, t.mean(), t.std()), color='#E07B39', lw=2.5,
        label=f'정규분포 (μ={t.mean():.1f}, σ={t.std():.1f})')
ax.set_xlabel('전비 (kWh/100km)'); ax.set_ylabel('밀도')
ax.set_title('타깃 변수(전비) 분포')
ax.legend(); plt.tight_layout(); plt.show()

print(f'범위: {t.min():.2f} ~ {t.max():.2f} kWh/100km  |  왜도: {t.skew():.3f}')

**해석** : 타깃이 왜도 0.03의 거의 완전한 정규분포다.
**판단** : 로그 변환·이상치 처리 없이 회귀 타깃으로 직접 사용한다.
범위 자체는 기획의 출발점(조건별 스프레드 345km)을 데이터로 재확인해준다.

### 2-2. 입력 변수 분포 — 어떤 방식으로 수집된 데이터인가?

In [ ]:
INPUT9 = ['speed_kmh', 'payload_kg', 'ambient_temp_C', 'hvac_power_kw', 'road_grade_pct',
          'battery_temp_C', 'driving_style_index', 'tire_pressure_bar', 'trip_distance_km']

fig, axes = plt.subplots(3, 3, figsize=(13, 8))
for ax, col in zip(axes.ravel(), INPUT9):
    ax.hist(df[col], bins=30, color='#5B7C99', alpha=0.85, edgecolor='white')
    ax.set_title(col, fontsize=10)
fig.suptitle('입력 변수 9종 분포', y=1.00)
plt.tight_layout(); plt.show()

print((df[INPUT9].mean() - (df[INPUT9].min() + df[INPUT9].max()) / 2).abs().round(2)
      .rename('평균과 범위중앙의 차이'))

**해석** : 9개 입력 전부가 각자의 물리 범위 안에서 **균등분포에 가깝다** —
평균이 범위 중앙과 거의 일치한다(속도 74/중앙 75, 적재 249/250, 경사 1.5/1.5).
자연 수집 데이터라면 속도·온도 등이 특정 구간에 몰리는 치우침이 나타나야 한다.

**판단** : 이 데이터는 조건 공간을 고르게 훑도록 설계된 **실험계획(DOE)형 데이터의 특성**을 보인다.
따라서 ① 희소 구간(혹한·급경사 등 극단 조건)도 충분히 학습되어 관측 범위 전반에서 예측이 안정적일 수 있고,
② 반대로 실도로의 조건 분포(고속도로 위주, 온화한 날씨 위주)와 다르므로 실환경 성능은 별도 검증이 필요하다(7장).

> 생성 코드나 문서를 직접 확인하지는 못했으므로, 여기서는 *분포가 DOE형 특성을 보인다*는
> 관찰 수준으로만 기술한다.

### 2-3. 변수 간 상관관계 히트맵 — 입력들은 서로 얽혀 있는가?

In [ ]:
cols = INPUT9 + ['energy_consumption_kwhper100km']
cm = df[cols].corr()

fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(cm, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=8)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f'{cm.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(cm.iloc[i, j]) > 0.5 else 'black')
plt.colorbar(im, shrink=0.8)
ax.set_title('변수 간 상관관계 히트맵')
plt.tight_layout(); plt.show()

off_diag = cm.loc[INPUT9, INPUT9].where(~np.eye(9, dtype=bool)).abs()
print(f'입력 변수 간 최대 |상관계수|: {off_diag.max().max():.3f}')

**해석** : 입력 변수들 **간의** 상관은 최대 |r|≈0.03으로 사실상 0이다.
(예: 공조전력과 외기온의 r≈-0.01. 실도로라면 추운 날 난방을 트니 상관이 생겼을 것)

**판단** : 다중공선성 문제가 관찰되지 않으므로, 회귀 계수는 **다른 입력 조건이 동일하다는 가정 아래
변수별 조건부 관계**를 비교적 안정적으로 나타낸다. 정량 확인은 2-8의 VIF로 마무리한다.

### 2-4. 변수별 타깃 상관계수 — 무엇이 전비를 움직이는가?

In [ ]:
corr = df.corr()['energy_consumption_kwhper100km'].drop('energy_consumption_kwhper100km')
corr = corr.sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ['#4A90D9' if v < 0 else '#E07B39' for v in corr]
ax.barh(corr.index, corr.values, color=colors)
for i, v in enumerate(corr.values):
    ax.text(v + (0.02 if v >= 0 else -0.02), i, f'{v:.2f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=9)
ax.axvline(0, color='gray', lw=0.8)
ax.set_xlim(-0.6, 0.7)
ax.set_xlabel('전비와의 상관계수'); ax.set_title('변수별 전비 상관계수')
plt.tight_layout(); plt.show()

**해석** : 핵심 동인은 **도로경사(+0.51) > 적재중량(+0.47) > 속도(+0.45) > 공조전력(+0.37)** 순.
위치에너지·관성·공기저항·냉난방 부하라는 물리 법칙과 일치한다.
`trip_distance_km`도 +0.17의 약한 양의 상관을 보인다(장거리일수록 전비가 다소 높음).

**판단** : 상위 동인 중 경사·속도는 **경로마다 달라지는 변수**다. 즉 경로 선택으로 절약 가능한 에너지가 실제로 크며,
"최소 에너지 경로 안내"라는 서비스의 존재 이유를 데이터가 뒷받침한다.
거리의 상관 0.17은 예측력이 있다는 뜻이지만, 이를 모델 입력으로 쓸지는 별도 요건 검토가 필요하다(3-2).

### 2-5. 입력-타깃 관계의 형태 — 선형인가, 비선형인가?

상관계수는 '선형' 관계의 세기만 잰다. 모델 계열 선택(선형 vs 트리)을 위해서는 **관계의 형태** 자체를 봐야 한다.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 8.5))
for ax, col in zip(axes.ravel(), INPUT9):
    b = pd.cut(df[col], 10)
    g = df.groupby(b, observed=True)['energy_consumption_kwhper100km'].mean()
    centers = [iv.mid for iv in g.index]
    hl = (col == 'ambient_temp_C')
    ax.plot(centers, g.values, marker='o', ms=3.5,
            color='#E07B39' if hl else '#5B7C99', lw=2 if hl else 1.5)
    ax.set_title(col + (' ← 비선형!' if hl else ''), fontsize=10,
                 color='#C0392B' if hl else 'black')
fig.suptitle('입력 변수별 구간 평균 전비 — 관계의 형태 점검', y=1.00)
plt.tight_layout(); plt.show()

**해석** : 9개 중 8개는 **거의 완전한 직선** 관계다(속도·경사·적재·공조·성향·거리는 우상향,
공기압·배터리온도는 완만한 우하향). 유일한 예외가 **외기온의 U자형**이다.

**판단** : "관계의 형태가 대부분 선형"이라는 관찰이 4장 모델 비교의 핵심 가설이 된다 —
*선형 모델 + 비선형 1곳만 피처로 보정*하는 전략이 복잡한 비선형 모델보다 유리할 수 있다.

### 2-6. 외기온의 U자형 관계 — 상관계수가 숨긴 패턴 (상세)

In [ ]:
bins = pd.cut(df['ambient_temp_C'], 8)
g = df.groupby(bins, observed=True)['energy_consumption_kwhper100km'].mean()
centers = [iv.mid for iv in g.index]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(centers, g.values, marker='o', color='#E07B39', lw=2)
ax.axvline(20, color='gray', ls='--', lw=1, label='쾌적온도 20°C')
ax.set_xlabel('외기온도 (°C)'); ax.set_ylabel('평균 전비 (kWh/100km)')
ax.set_title('외기온도 구간별 평균 전비 — U자형 비선형 관계')
ax.legend(); plt.tight_layout(); plt.show()

**해석** : 외기온의 선형 상관계수는 -0.16으로 약해 보이지만, 구간별 평균은 **영하에서 최고(≈25.7),
온화한 구간에서 최저(≈23.5), 폭염에서 재상승**하는 U자형이다. 저온에서는 난방·배터리 열관리,
고온에서는 냉방 부하가 커지기 때문으로, 선형 상관계수가 이 효과를 과소평가하고 있었다.

**판단** : 쾌적온도(20°C)와의 편차를 파생변수로 만들어 선형 모델도 이 U자형을 학습할 수 있게 한다.
편차의 **형태**(절대값 vs 제곱)는 4-6에서 검증으로 결정한다.

### 2-7. 도로경사와 전비 — 최대 동인의 형태 (상세)

In [ ]:
s = df.sample(1200, random_state=SEED)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(s['road_grade_pct'], s['energy_consumption_kwhper100km'],
           s=12, alpha=0.35, color='#5B7C99')
z = np.polyfit(df['road_grade_pct'], df['energy_consumption_kwhper100km'], 1)
xs = np.linspace(df['road_grade_pct'].min(), df['road_grade_pct'].max(), 50)
ax.plot(xs, np.polyval(z, xs), color='#E07B39', lw=2.5,
        label=f'추세선: 경사 1%p당 +{z[0]:.2f} kWh/100km')
ax.set_xlabel('도로 경사 (%)'); ax.set_ylabel('전비 (kWh/100km)')
ax.set_title('도로 경사 vs 전비 (표본 1,200건)')
ax.legend(); plt.tight_layout(); plt.show()

**해석** : 경사-전비 관계는 뚜렷한 **선형**이며, 내리막(-5%)에서는 회생제동 효과로 전비가 낮아지는
패턴까지 이어진다. 경사 1%p당 약 +0.5 kWh/100km.
**판단** : 최대 동인이 선형 형태라는 것은 선형 계열 모델이 유리할 수 있다는 신호다. 4장에서 검증한다.

### 2-8. 다중공선성 정량 점검 (VIF)

2-3의 히트맵 관찰을 VIF(분산팽창계수)로 정량 확인한다. 통상 VIF > 10 이면 공선성 문제로 본다.

In [ ]:
def vif_table(X):
    """각 변수를 나머지 변수로 회귀한 R²로 VIF = 1/(1-R²) 계산"""
    out = {}
    for c in X.columns:
        others = [k for k in X.columns if k != c]
        r2 = LinearRegression().fit(X[others], X[c]).score(X[others], X[c])
        out[c] = 1 / (1 - r2)
    return pd.Series(out, name='VIF').round(3).sort_values(ascending=False)

vif_table(df[INPUT9])

**해석** : 전 변수 VIF ≈ 1.00 (완전 독립 시의 이론적 최솟값)으로 다중공선성 문제는 관찰되지 않았다.
**판단** : 회귀 계수의 분산 팽창이 없으므로 계수를 변수별 조건부 기여로 안정적으로 읽을 수 있고(5-5),
공선성 대응(변수 제거, PCA, 강한 정규화)은 불필요하다.

---
### EDA 종합 — 분석에서 도출한 판단 5가지

| # | 관찰 | 판단 (→ 반영 위치) |
|---|---|---|
| 1 | 타깃이 정규분포, 이상치 없음 | 변환 없이 직접 회귀 (→ 3장) |
| 2 | 입력이 균등분포 (DOE형 특성) | 관측 범위 전반 학습 가능 / 실환경 재검증 필요 (→ 7장) |
| 3 | 입력 간 상관 ≈ 0, VIF ≈ 1 | 계수의 조건부 해석 가능, 공선성 대응 불필요 (→ 5-5) |
| 4 | 관계 형태: 8개 선형 + 외기온 U자형 | 선형 모델 + 온도 파생변수 전략 (→ 3·4장) |
| 5 | 경사·속도가 상위 동인 / 거리도 r=0.17 | 경로 간 에너지 차이 실존(→6장) / 거리 사용 방식 검토(→3-2) |

---
## 3. 전처리

정제할 결함이 없음을 1장에서 확인했으므로, 전처리의 핵심은
**① 거리 변수의 역할 결정 ② 파생변수 생성 ③ 검증 체계 설계**다.

### 3-1. 파생변수 생성 — 물리 근거가 있는 것만

임의 조합이 아니라 물리 법칙에 근거한 후보를 만들고, **4장에서 기여를 검증한 뒤 채택 여부를 결정**한다.

| 파생변수 | 정의 | 물리 근거 |
|---|---|---|
| `temp_discomfort` | \|외기온 − 20\| | 쾌적온도에서 멀수록 공조·열관리 부하 증가 (2-6) — V자형 |
| `temp_dev_sq` | (외기온 − 20)² | 같은 근거, 편차에 **가속적**으로 반응 — 포물선형 |
| `grade_x_payload` | 경사 × 적재중량 | 등판 저항 = 중량 × 경사 (곱셈 관계) |
| `speed_sq` | 속도² | 공기저항은 속도 제곱에 비례 |

파생변수는 **행 단위 연산**(다른 행의 정보를 쓰지 않음)이므로 분할 이전에 생성해도 데이터 누수가 없다.

In [ ]:
df['temp_discomfort']  = (df['ambient_temp_C'] - 20).abs()
df['temp_dev_sq']      = (df['ambient_temp_C'] - 20) ** 2
df['grade_x_payload']  = df['road_grade_pct'] * df['payload_kg']
df['speed_sq']         = df['speed_kmh'] ** 2

TARGET = 'energy_consumption_kwhper100km'
DISTANCE_COL = 'trip_distance_km'          # 모델 입력이 아닌 '변환 계수' (3-2)

# 전비 예측에 사용할 원본 입력 (거리 제외)
ORIG = ['speed_kmh', 'payload_kg', 'ambient_temp_C', 'hvac_power_kw',
        'road_grade_pct', 'battery_temp_C', 'driving_style_index', 'tire_pressure_bar']
DERIVED = ['temp_discomfort', 'temp_dev_sq', 'grade_x_payload', 'speed_sq']

print('원본 입력:', len(ORIG), '개  |  파생 후보:', len(DERIVED), '개  |  변환 계수:', DISTANCE_COL)

### 3-2. 설계 결정 — 거리를 전비 모델의 입력에서 제외한다

2-4에서 `trip_distance_km`는 전비와 **+0.17의 상관**을 보였고, 실제로 입력에 넣으면 성능이 오른다.
그럼에도 제외하는 이유는 **경로 비용 함수로서의 요건** 때문이다.

#### 왜 문제가 되는가 — 거리 제곱항의 발생

전비 예측식을 단순화하면 링크 $i$의 예측 전비는

$$\hat{y}_i = a + \beta d_i + (\text{기타 항})$$

이고, 링크 에너지는 여기에 거리를 곱해 얻는다.

$$E_i = \hat{y}_i \times \frac{d_i}{100} = \frac{a\,d_i + \beta\,d_i^{2} + \cdots}{100}$$

거리 항이 **제곱**으로 들어간다. 따라서 동일한 100km 구간을
1개 링크로 표현하면 거리항이 $\beta \cdot 100^2$이지만,
50km 2개 링크로 나누면 $2 \times \beta \cdot 50^2 = \beta \cdot 5{,}000$ 이 되어 값이 달라진다.

**물리적으로 같은 경로인데 그래프를 어떻게 쪼갰느냐에 따라 비용이 바뀐다** — 분할 불변성 위반이다.
표준 경로탐색에 사용할 링크 비용은 그래프의 임의적인 분할 방식에 민감해서는 안 된다.

아래에서 이 현상을 실제로 측정한다.

In [ ]:
# 거리를 입력에 포함한 모델로, 동일 경로를 여러 방식으로 분할해 총에너지를 비교
demo_feats = ORIG + [DISTANCE_COL, 'temp_dev_sq']
demo_pipe = Pipeline([('s', StandardScaler()), ('m', LinearRegression())]).fit(df[demo_feats], df[TARGET])

cond = dict(speed_kmh=80, payload_kg=300, ambient_temp_C=10, hvac_power_kw=2.0,
            road_grade_pct=1.0, battery_temp_C=25, driving_style_index=0.5, tire_pressure_bar=2.4)
cond['temp_dev_sq'] = (cond['ambient_temp_C'] - 20) ** 2

rows = []
for n in [1, 2, 5, 10, 20]:
    d = 100 / n
    seg = pd.DataFrame([{**cond, DISTANCE_COL: d} for _ in range(n)])
    total = (demo_pipe.predict(seg[demo_feats]) * d / 100).sum()
    rows.append({'링크 수': n, '링크당 거리(km)': d, '총 에너지(kWh)': round(total, 4)})

split_demo = pd.DataFrame(rows)
split_demo['1링크 대비 차이(%)'] = ((split_demo['총 에너지(kWh)'] / split_demo['총 에너지(kWh)'].iloc[0] - 1) * 100).round(2)
split_demo

**측정 결과** : 동일한 100km 경로가 1개 링크일 때 24.18kWh, 20개 링크일 때 23.23kWh로 **약 4% 차이**가 난다.
링크 분할은 지도 데이터의 표현 방식일 뿐 물리적 실체가 아니므로, 이 차이는 순수한 모델링 오류다.

#### 결정과 그 대가

거리를 제외하면 예측 성능은 떨어진다(4-2에서 정량 측정). 그럼에도
**경로 비용 함수의 분할 불변성과 가법성**을 우선해 거리를 전비 모델 입력에서 제외하고,
예측 전비를 총에너지로 변환하는 **결정론적 계산에만** 사용한다.

$$E_{\text{경로}} = \sum_i \hat{y}_i \times \frac{d_i}{100}, \qquad \hat{y}_i = f(\text{조건}_i) \;\; (d_i \notin \text{입력})$$

이제 $\hat{y}_i$가 거리와 무관하므로 $E$는 링크 분할 방식에 불변이다(6-2에서 자동 검증).

### 3-3. Train / Validation / Test 분할 (60 / 20 / 20)

- **Train(60%)** : 모델 학습
- **Validation(20%)** : 피처 구성·모델·하이퍼파라미터 등 **모든 개발 판단**
- **Test(20%)** : 최종 모델의 성능 확인

스케일러는 **Train에서만 fit**하고 Val/Test에는 transform만 적용한다.

In [ ]:
y = df[TARGET]
X_tr, X_tmp, y_tr, y_tmp = train_test_split(df, y, test_size=0.4, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=SEED)
print(f'Train {len(X_tr)} / Val {len(X_val)} / Test {len(X_te)}')

---
## 4. 모델 설계 및 비교

**Baseline → Compare → Improve**의 3단 구조. Improve를 5겹으로 깊게 한다:
① 거리 변수 영향 정량화 → ② 파생변수 기여 분리 → ③ 전 계열 튜닝 →
④ 기계적 전수 탐색으로 성능 상한 확인 → ⑤ 파생변수 형태 재검증.

**모든 비교와 판단은 Validation 세트에서 수행한다.** Test는 5장에서 최종 확인용으로만 사용한다.

### 4-1. Baseline + Compare : 원본 8개 피처, 기본 파라미터

In [ ]:
def evaluate(model, feats, name):
    """Train으로 학습, Validation으로 평가. 추론속도 100회 평균, Train R²로 과적합 갭 확인."""
    scaler = StandardScaler().fit(X_tr[feats])
    m = model.fit(scaler.transform(X_tr[feats]), y_tr)
    Xv = scaler.transform(X_val[feats])
    pred = m.predict(Xv)
    t0 = time.perf_counter()
    for _ in range(100):
        m.predict(Xv)
    ms = (time.perf_counter() - t0) / 100 * 1000
    return {'모델': name, 'Train R²': r2_score(y_tr, m.predict(scaler.transform(X_tr[feats]))),
            'Val R²': r2_score(y_val, pred), 'MAE': mean_absolute_error(y_val, pred),
            'RMSE': np.sqrt(mean_squared_error(y_val, pred)), '추론(ms/1600건)': ms}

results = []
results.append(evaluate(LinearRegression(), ORIG, 'Linear Regression (베이스라인)'))
results.append(evaluate(Ridge(alpha=1.0), ORIG, 'Ridge (α=1)'))
results.append(evaluate(Lasso(alpha=0.1), ORIG, 'Lasso (α=0.1)'))
results.append(evaluate(DecisionTreeRegressor(random_state=SEED), ORIG, 'Decision Tree'))
results.append(evaluate(RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1), ORIG, 'Random Forest'))
results.append(evaluate(XGBRegressor(random_state=SEED, n_jobs=-1), ORIG, 'XGBoost'))
results.append(evaluate(LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1), ORIG, 'LightGBM'))
pd.DataFrame(results).round(4)

**해석** : 기본 설정 기준으로 **선형 회귀가 트리 계열을 앞선다.**
2-5에서 확인한 "관계 대부분이 선형"이라는 구조 때문이다. 트리 계열은 연속적 선형 관계를
계단식으로 근사하느라 손해를 본다. 다만 기본 설정만으로 탈락시키는 것은 불공정하므로 4-4에서 튜닝 후 재평가한다.

### 4-2. Improve ① : 거리 변수 제외의 대가 정량화

3-2의 결정이 성능 면에서 얼마를 지불하는지 Validation에서 측정한다.

In [ ]:
def eval_linear(feats, label):
    scaler = StandardScaler().fit(X_tr[feats])
    m = LinearRegression().fit(scaler.transform(X_tr[feats]), y_tr)
    pred = m.predict(scaler.transform(X_val[feats]))
    return {'피처 구성': label, 'Val R²': r2_score(y_val, pred), 'MAE': mean_absolute_error(y_val, pred)}

dist_cost = pd.DataFrame([
    eval_linear(ORIG + [DISTANCE_COL, 'temp_dev_sq'], '거리 포함 (9개 + 온도파생)'),
    eval_linear(ORIG + ['temp_dev_sq'],               '거리 제외 (8개 + 온도파생) ★ 채택'),
]).round(4)
dist_cost['R² 차이'] = (dist_cost['Val R²'] - dist_cost['Val R²'].iloc[0]).round(4)
print(dist_cost.to_string(index=False))

# 거리 제거가 나머지 계수를 흔드는지 점검
X_trval = pd.concat([X_tr, X_val]); y_trval = pd.concat([y_tr, y_val])
c_with = pd.Series(LinearRegression().fit(X_trval[ORIG + [DISTANCE_COL, 'temp_dev_sq']], y_trval).coef_,
                   index=ORIG + [DISTANCE_COL, 'temp_dev_sq']).drop(DISTANCE_COL)
c_without = pd.Series(LinearRegression().fit(X_trval[ORIG + ['temp_dev_sq']], y_trval).coef_,
                      index=ORIG + ['temp_dev_sq'])
coef_shift = pd.DataFrame({'거리 포함': c_with, '거리 제외': c_without})
coef_shift['변화율(%)'] = ((coef_shift['거리 제외'] - coef_shift['거리 포함']) / coef_shift['거리 포함'].abs() * 100)
coef_shift.round(4)

**해석** : 거리를 빼면 Validation R²가 **0.952 → 0.929로 약 0.023 감소**한다(MAE 0.66 → 0.80).
분명한 대가다. 그러나 3-2에서 본 4%의 분할 의존성은 서비스로서 허용할 수 없는 종류의 오류이므로,
**설명력 약 2.3%p를 지불하고 분할 불변성을 선택**한다.

계수 변화율은 최대 1.8%로 작다. 이는 본 합성데이터에서 거리가 다른 입력과 거의 독립적이어서
(2-3, VIF≈1), 거리 제거가 **관측된 나머지 계수**에 미치는 영향이 제한적임을 보여준다.
(관측되지 않은 변수까지 포함한 일반적 주장은 아니다.)

### 4-3. Improve ② : 파생변수 기여 검증 (Ablation)

In [ ]:
ablation = pd.DataFrame([
    eval_linear(ORIG, '원본 8개'),
    eval_linear(ORIG + ['temp_discomfort'], '+ temp_discomfort'),
    eval_linear(ORIG + ['grade_x_payload'], '+ grade_x_payload'),
    eval_linear(ORIG + ['speed_sq'], '+ speed_sq'),
    eval_linear(ORIG + ['temp_discomfort', 'grade_x_payload', 'speed_sq'], '+ 전체 3개'),
]).round(4)
ablation

**해석** : 개선은 사실상 **온도 파생변수 하나가 만든다**.
`grade_x_payload`·`speed_sq`는 기여가 없다 — 이 데이터에서 경사 효과는 중량과 가법적이고,
속도 효과는 관측 범위(20~130km/h)에서 거의 선형이기 때문이다(2-5의 형태 관찰과 일치).
**판단** : 기여 없는 두 후보는 폐기한다. 온도 파생변수의 **형태**는 4-6에서 결정한다.

### 4-4. Improve ③ : 전 계열 하이퍼파라미터 튜닝

기본 설정 비교만으로 결론 내리지 않기 위해, 5개 계열 전부에 GridSearchCV(교차검증)를 적용한다.

In [ ]:
tune_log = []
sc = StandardScaler().fit(X_tr[ORIG])

for name, Est, grid in [('Ridge', Ridge, {'alpha': [0.01, 0.1, 1, 10, 100]}),
                        ('Lasso', Lasso, {'alpha': [0.001, 0.01, 0.1, 1]})]:
    gs = GridSearchCV(Est(), grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    gs.fit(sc.transform(X_tr[ORIG]), y_tr)
    tune_log.append(f'{name}: {gs.best_params_}')
    results.append(evaluate(Est(**gs.best_params_), ORIG, f'{name} (Tuned)'))

gs = GridSearchCV(DecisionTreeRegressor(random_state=SEED),
                  {'max_depth': [6, 8, 10, 12, None], 'min_samples_leaf': [1, 5, 20, 50]},
                  cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
gs.fit(sc.transform(X_tr[ORIG]), y_tr)
tune_log.append(f'DecisionTree: {gs.best_params_}')
results.append(evaluate(DecisionTreeRegressor(random_state=SEED, **gs.best_params_), ORIG, 'Decision Tree (Tuned)'))

gs = GridSearchCV(RandomForestRegressor(random_state=SEED, n_jobs=-1),
                  {'n_estimators': [200, 400], 'max_depth': [15, None], 'max_features': ['sqrt', 0.5]},
                  cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
gs.fit(sc.transform(X_tr[ORIG]), y_tr)
tune_log.append(f'RandomForest: {gs.best_params_}')
results.append(evaluate(RandomForestRegressor(random_state=SEED, n_jobs=-1, **gs.best_params_), ORIG, 'Random Forest (Tuned)'))

gs = GridSearchCV(XGBRegressor(random_state=SEED, n_jobs=-1),
                  {'n_estimators': [200, 400], 'max_depth': [3, 5, 7],
                   'learning_rate': [0.05, 0.1], 'subsample': [0.8, 1.0]},
                  cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
gs.fit(sc.transform(X_tr[ORIG]), y_tr)
tune_log.append(f'XGBoost: {gs.best_params_}')
results.append(evaluate(XGBRegressor(random_state=SEED, n_jobs=-1, **gs.best_params_), ORIG, 'XGBoost (Tuned)'))
# 공정성: 최적 XGBoost에 온도 파생변수도 제공
results.append(evaluate(XGBRegressor(random_state=SEED, n_jobs=-1, **gs.best_params_),
                        ORIG + ['temp_dev_sq'], 'XGBoost (Tuned +temp_dev_sq)'))

gs = GridSearchCV(LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1),
                  {'n_estimators': [200, 400], 'max_depth': [3, 5, 7],
                   'learning_rate': [0.05, 0.1], 'subsample': [0.8, 1.0]},
                  cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
gs.fit(sc.transform(X_tr[ORIG]), y_tr)
tune_log.append(f'LightGBM: {gs.best_params_}')
results.append(evaluate(LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1, **gs.best_params_), ORIG, 'LightGBM (Tuned)'))
# 공정성: 최적 LightGBM에 온도 파생변수도 제공
results.append(evaluate(LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1, **gs.best_params_),
                        ORIG + ['temp_dev_sq'], 'LightGBM (Tuned +temp_dev_sq)'))

print('\n'.join(tune_log))
pd.DataFrame(results).round(4)

**해석** :
- **XGBoost는 튜닝으로 개선**된다. 최적 조합이 얕은 트리(depth 3) 다수 + 낮은 학습률이라는 점이 시사적이다 —
  선형에 가까운 구조를 부드럽게 근사하는 방향으로 수렴했다. 온도 파생변수를 줘도 선형 모델에는 미치지 못한다.
- **정규화는 이득을 주지 못한다** : 교차검증이 고른 Ridge α(=10)를 적용해도 Validation R²는 α=1일 때와
  소수 넷째 자리에서만 다르고, Lasso는 최적 α가 0.001로 사실상 정규화를 끄는 방향이다.
  표본(4,800) 대비 피처(8)가 적고 2-8에서 확인한 대로 공선성이 없어, 선형 모델에 과적합 여지 자체가 작기 때문이다.
- **Decision Tree는 튜닝해도 부진**: 단일 트리의 계단식 근사 한계는 파라미터로 극복되지 않는다.

### 4-5. Improve ④ : 기계적 전수 탐색으로 성능 상한 확인 (2차 다항)

"도메인 지식으로 만든 파생변수가 놓친 비선형·상호작용이 더 있는가?"를 확인하기 위해,
8개 변수의 **모든 제곱·상호작용항**을 기계적으로 생성해 Ridge로 적합한다.
이 성능이 곧 '2차 관계까지 허용했을 때의 상한선'이다.

In [ ]:
poly_pipe = Pipeline([('s', StandardScaler()),
                      ('p', PolynomialFeatures(2, include_bias=False)),
                      ('m', Ridge())])
gs = GridSearchCV(poly_pipe, {'m__alpha': [0.1, 1, 10, 100]}, cv=5,
                  scoring='neg_mean_absolute_error', n_jobs=-1)
gs.fit(X_tr[ORIG], y_tr)
pred = gs.predict(X_val[ORIG])
n_terms = PolynomialFeatures(2, include_bias=False).fit(X_tr[ORIG]).n_output_features_
results.append({'모델': f'Poly-2 Ridge ({n_terms}항, 상한 확인용)',
                'Train R²': r2_score(y_tr, gs.predict(X_tr[ORIG])),
                'Val R²': r2_score(y_val, pred), 'MAE': mean_absolute_error(y_val, pred),
                'RMSE': np.sqrt(mean_squared_error(y_val, pred)), '추론(ms/1600건)': np.nan})
print(f'Poly-2 Ridge ({n_terms}항): Val R² = {results[-1]["Val R²"]:.4f}, MAE = {results[-1]["MAE"]:.4f}')

names = gs.best_estimator_.named_steps['p'].get_feature_names_out(ORIG)
coefs = pd.Series(gs.best_estimator_.named_steps['m'].coef_, index=names)
second = coefs[[n for n in names if ' ' in n or '^2' in n]]
second.reindex(second.abs().sort_values(ascending=False).index).head(8).round(4)

**해석** : 2차·상호작용항 중 유의미한 계수는 **`ambient_temp_C²` 단 하나**이고, 2위부터는 노이즈 수준이다.
**판단** : 이 데이터에 숨은 비선형은 **온도 하나뿐**임이 전수 탐색으로 확정됐다.
전 항을 다 쓸 필요 없이 온도 항 하나만 제대로 넣으면 상한에 도달할 수 있다.
다만 유효한 항이 절대값(|t−20|)이 아니라 **제곱 형태**라는 점이 4-3의 후보와 다르다 → 형태를 재검증한다.

### 4-6. Improve ⑤ : 온도 파생변수의 형태 재검증 — V자 vs 포물선

In [ ]:
shape_test = pd.DataFrame([
    eval_linear(ORIG + ['temp_discomfort'], '|t-20|   (V자형)'),
    eval_linear(ORIG + ['temp_dev_sq'],     '(t-20)²  (포물선형)'),
    eval_linear(ORIG + ['temp_discomfort', 'temp_dev_sq'], '둘 다 (10피처)'),
]).round(4)
print(shape_test.to_string(index=False))

FINAL_FEATS = ORIG + ['temp_dev_sq']
results.append(evaluate(LinearRegression(), FINAL_FEATS, 'Linear (+temp_dev_sq) ★'))
print('\n최종 피처', len(FINAL_FEATS), '개:', FINAL_FEATS)

**해석** : **포물선형이 V자형을 앞서고, 44항 다항 전수 탐색과도 동등하거나 근소하게 앞선다**
(선형+포물선 0.9287 vs Poly-2 0.9279). 둘을 같이 넣어도 추가 이득이 거의 없다 — 포물선이 실제 온도-부하 관계(쾌적 온도에서 멀어질수록
가속적으로 커지는 공조 부하)에 더 가까운 형태다.
**판단** : 최종 피처를 **원본 8개 + `(t−20)²` = 9개**로 확정한다.
전수 탐색이 찾은 상한 성능을 해석 가능한 9개 피처로 재현한 것이다.

### 4-7. 과적합 진단 — Train/Validation 갭과 학습곡선

In [ ]:
diag = pd.DataFrame(results)[['모델', 'Train R²', 'Val R²']].copy()
diag['갭 (Train-Val)'] = diag['Train R²'] - diag['Val R²']
diag.round(4).sort_values('갭 (Train-Val)', ascending=False)

In [ ]:
sizes, tr_scores, cv_scores = learning_curve(
    Pipeline([('s', StandardScaler()), ('m', LinearRegression())]),
    X_tr[FINAL_FEATS], y_tr, cv=5, scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sizes, tr_scores.mean(1), 'o-', color='#5B7C99', label='Train R²')
ax.fill_between(sizes, cv_scores.mean(1) - cv_scores.std(1), cv_scores.mean(1) + cv_scores.std(1),
                alpha=0.15, color='#E07B39')
ax.plot(sizes, cv_scores.mean(1), 'o-', color='#E07B39', label='교차검증 R² (±1σ)')
ax.set_xlabel('학습 표본 수'); ax.set_ylabel('R²')
ax.set_title('최종 모델 학습곡선 — 과적합·데이터 충분성 진단')
ax.legend(); plt.tight_layout(); plt.show()

**해석** :
- **갭 표** : Decision Tree·Random Forest 계열은 Train에서만 좋은 전형적 과적합 패턴을 보인다.
  **최종 선형 모델의 갭은 0.003 수준**으로, 현재 표본 수(4,800)와 피처 수(9) 조건에서는
  뚜렷한 과적합 징후가 관찰되지 않는다.
- **학습곡선** : Train과 교차검증 곡선이 이미 수렴했고 표본 2,000건 근처부터 성능이 포화된다.
  8,000건은 이 문제에 충분하며, 다음 과제는 데이터 양이 아니라 실환경 다양성 확보(7장)임을 시사한다.

### 4-8. 종합 비교 및 최종 선정

In [ ]:
compare = pd.DataFrame(results).round(4).sort_values('Val R²', ascending=False).reset_index(drop=True)
print(f'총 {len(compare)}개 구성 비교')

fig, ax = plt.subplots(figsize=(9.5, 6.5))
comp = compare.sort_values('Val R²')
def color_of(n):
    if '★' in n: return '#E07B39'
    if 'Poly' in n: return '#C9A66B'
    if any(k in n for k in ['Linear', 'Ridge', 'Lasso']): return '#B0B0B0'
    return '#5B7C99'
ax.barh(comp['모델'], comp['Val R²'], color=[color_of(n) for n in comp['모델']])
for i, v in enumerate(comp['Val R²']):
    ax.text(v + 0.004, i, f'{v:.3f}', va='center', fontsize=8.5)
ax.set_xlim(0.65, 1.02)
ax.set_xlabel('Validation R²')
ax.set_title(f'전체 {len(compare)}개 구성 Validation R² 비교 (★ 최종 선정)')
plt.tight_layout(); plt.show()
compare

### 최종 선정 — 4축 종합 평가

| 평가축 | **Linear (+temp_dev_sq) ★** | XGBoost (Tuned) | Poly-2 Ridge |
|---|---|---|---|
| 정확도 (Val R²) | **1위 (0.929)** | 3위 (0.920) | 2위 (0.928) |
| 추론 속도 | **~0.04ms (가장 빠름)** | 약 5.4ms (100배 이상) | 중간 |
| 해석 가능성 | **9개 계수 = 물리량 단위 기여도** | 중요도만 산출 | 다수 항 — 해석 곤란 |
| 과적합 갭 | **0.003** | 0.029 | 0.005 |
| 임베디드 적합성 | **곱셈-덧셈 9회** | 트리 수백 개 저장 | 항 다수 연산 |

**판단** : 정확도 1위이면서 가장 단순한 구성을 선택한다(파시모니 원칙).
다항 전수 탐색이 보여준 상한을 도메인 피처 1개로 재현했으므로 추가 복잡도를 감수할 이유가 없다.
**데이터 구조를 EDA로 진단하고 → 튜닝·전수 탐색으로 상한을 확인한 뒤 → 그 상한을 최소 복잡도로
달성하는 모델을 선택**했다는 것이 본 분석의 결론이다.

---
## 5. 최종 모델 평가

### 5-1. 주 성능 지표 — 반복 교차검증

단일 분할 결과는 분할 운(luck)에 좌우될 수 있다. 최종 모델의 성능은
**Train+Val(6,400건)에 대한 5-fold × 4회 반복 교차검증(총 20폴드)** 으로 보고하고,
Test는 5-2에서 확인용으로 한 번 사용한다.

In [ ]:
X_trval = pd.concat([X_tr, X_val]); y_trval = pd.concat([y_tr, y_val])
final_pipe = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])

rkf = RepeatedKFold(n_splits=5, n_repeats=4, random_state=SEED)
cv_r2  = cross_val_score(final_pipe, X_trval[FINAL_FEATS], y_trval, cv=rkf, scoring='r2')
cv_mae = -cross_val_score(final_pipe, X_trval[FINAL_FEATS], y_trval, cv=rkf,
                          scoring='neg_mean_absolute_error')
print(f'[반복 교차검증 20폴드]  R² = {cv_r2.mean():.4f} ± {cv_r2.std():.4f}'
      f'   MAE = {cv_mae.mean():.4f} ± {cv_mae.std():.4f}')

**해석** : 20개 폴드에 걸쳐 표준편차가 매우 작아(R² ±0.003) 성능이 분할에 거의 좌우되지 않는다.
단일 분할 하나보다 신뢰할 수 있는 성능 추정치다.

### 5-2. 홀드아웃 Test 확인

In [ ]:
final_pipe.fit(X_trval[FINAL_FEATS], y_trval)
pred_te = final_pipe.predict(X_te[FINAL_FEATS])
r2   = r2_score(y_te, pred_te)
mae  = mean_absolute_error(y_te, pred_te)
rmse = np.sqrt(mean_squared_error(y_te, pred_te))
print(f'[Test 확인]  R² = {r2:.4f}  |  MAE = {mae:.3f}  |  RMSE = {rmse:.3f} (kWh/100km)')
print(f'[교차검증]   R² = {cv_r2.mean():.4f}  |  MAE = {cv_mae.mean():.3f}')

**해석** : Test 결과가 교차검증 추정치와 일치한다(R² 차이 0.01 이내).
두 방식이 같은 값을 가리키므로 성능 추정이 안정적이다.

> **투명성 고지** : 본 프로젝트에서 거리 변수 제외 결정은 Validation 성능과 분할 불변성 요건에 근거했으나,
> 분석 과정에서 Test 성능도 관찰했다. 따라서 Test를 완전히 독립적인 홀드아웃으로 보기는 어렵고,
> 위 반복 교차검증을 주 지표로 삼는다. 엄밀한 외부 검증은 7장의 향후 과제로 남긴다.

### 5-3. 실제값 vs 예측값 — 모델 계열별 형태 비교

In [ ]:
def fit_predict(model, feats):
    p = Pipeline([('s', StandardScaler()), ('m', model)]).fit(X_trval[feats], y_trval)
    return p.predict(X_te[feats])

preds = {
    'Lasso (원본 8피처)': fit_predict(Lasso(alpha=0.1), ORIG),
    'XGBoost (원본 8피처)': fit_predict(XGBRegressor(random_state=SEED, n_jobs=-1), ORIG),
    'LightGBM (원본 8피처)': fit_predict(LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1), ORIG),
    'Linear +temp_dev_sq (최종)': pred_te,
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4.4), sharex=True, sharey=True)
palette = ['#7B8FA8', '#D98E5F', '#C9A66B', '#5BA57F']
for ax, (name, p), c in zip(axes, preds.items(), palette):
    ax.scatter(y_te, p, s=10, alpha=0.35, color=c)
    lims = [y_te.min(), y_te.max()]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_title(f'{name}\nR²={r2_score(y_te, p):.3f}, MAE={mean_absolute_error(y_te, p):.2f}')
    ax.set_xlabel('실제값 (kWh/100km)')
axes[0].set_ylabel('예측값 (kWh/100km)')
plt.tight_layout(); plt.show()

**해석** : 최종 모델의 점들이 대각선(완전 일치선)에 가장 조밀하게 붙는다.
XGBoost는 저전비·고전비 극단에서 예측이 중심으로 쏠리는 수축 경향이 보인다.

### 5-4. 잔차의 구간별 균일성

내비게이션 관점의 질문: 오차가 고전비 구간(겨울 등판 = 잔량이 가장 불안한 순간)에 몰려 있지는 않은가.

In [ ]:
seg = pd.DataFrame({'실제': y_te.values, '절대오차': np.abs(y_te.values - pred_te)})
seg_mae = seg.groupby(pd.cut(seg['실제'], [11, 18, 22, 26, 30, 35]), observed=True)['절대오차'] \
             .agg(['mean', 'count']).round(3)
seg_mae.columns = ['구간 MAE', '표본수']
print(seg_mae)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(i) for i in seg_mae.index], seg_mae['구간 MAE'], color='#5BA57F', alpha=0.85)
ax.axhline(mae, color='#E07B39', ls='--', label=f'전체 MAE = {mae:.2f}')
ax.set_xlabel('실제 전비 구간 (kWh/100km)'); ax.set_ylabel('구간별 MAE')
ax.set_title('전비 구간별 예측 오차')
ax.legend(); plt.tight_layout(); plt.show()

**해석** : 구간별 MAE가 전체 평균 주변에 비교적 균일하게 분포한다.
**단, 이는 본 합성데이터의 관측 범위와 고정 분할 조건에서의 결과**다.
실제 차량·운전자·계절·경로가 달라지는 환경에서는 별도의 외부 검증이 필요하다.

### 5-5. 회귀 계수 해석 — 모델이 곧 비용 함수 설계서

해석 전용으로 스케일링 없이 재적합하여 **자연 단위 그대로의 계수**를 읽는다.

In [ ]:
interp = LinearRegression().fit(X_trval[FINAL_FEATS], y_trval)
coef = pd.DataFrame({'계수': interp.coef_}, index=FINAL_FEATS).round(5)
c = interp.coef_
coef['해석 (다른 조건이 동일할 때)'] = [
    f'속도 +10km/h → 전비 {c[0] * 10:+.2f}',
    f'적재 +100kg → 전비 {c[1] * 100:+.2f}',
    '온도 선형항 (아래 2차항과 결합 해석)',
    f'공조 +1kW → 전비 {c[3]:+.2f}',
    f'경사 +1%p → 전비 {c[4]:+.2f}',
    f'배터리온도 +1°C → 전비 {c[5]:+.3f}',
    f'급가감속 성향 0→1 → 전비 {c[6]:+.2f}',
    f'공기압 +0.1bar → 전비 {c[7] * 0.1:+.2f}',
    '온도 편차 제곱항 (아래 결합 효과 참조)',
]
print(f'절편: {interp.intercept_:.2f}')
tl = coef.loc['ambient_temp_C', '계수']; tq = coef.loc['temp_dev_sq', '계수']
print('\n[온도 결합 효과] 20°C 기준 대비 전비 변화')
for t in [-10, 0, 35]:
    print(f'  {t:>4}°C : {tl * (t - 20) + tq * (t - 20) ** 2:+.2f} kWh/100km')
coef

**해석** : 계수 하나하나가 업무 언어로 번역된다. 아래는 모두 *다른 입력 조건이 동일할 때
모델이 예측하는 차이*이며, 실제 개입 효과를 확정하려면 실차 실험이 필요하다.

- **경사 +1%p → 전비 +0.50** : HD맵 고도 기반 경로 비용 가중치의 근거
- **온도 결합 효과 — 혹한 -10°C는 쾌적(20°C) 대비 +3.2, 0°C는 +1.5, 폭염 35°C는 +0.5** :
  겨울철 주행거리 불안(저온이 고온보다 훨씬 불리한 비대칭)의 정량 근거이자 계절별 안내 보정의 근거
- **급가감속 성향(0→1) → +3.5** : 운전자 프로파일별 개인화 안내의 근거
- **공조 +1kW → +0.90** : 공조 소비전력이 1kW 낮은 사례는 전비가 약 0.90 낮게 예측된다는 뜻이며,
  절전 코칭 기능의 출발점이 된다
- **공기압 +0.1bar → -0.15** : TPMS 저압 경고를 에너지 관점으로 확장할 근거

트리 모델의 특성 중요도는 "무엇이 중요한가"까지만 답하지만, 선형 계수는 **"얼마나, 어느 방향으로"** 까지 답한다.
이것이 해석 가능성을 선정 기준에 넣은 이유다.

### 5-6. 주행가능거리 오차로 환산 — 평균이 아니라 분포로

$\text{주행가능거리} = \text{배터리 용량} / \text{전비} \times 100$ 은 전비의 **역수**다.
평균 전비 한 점에서 환산하면 비선형성 때문에 실제 오차를 과소평가하므로, **표본별로 계산**한다.

In [ ]:
BATTERY = 60  # kWh

actual_range_km    = BATTERY / y_te.to_numpy() * 100
predicted_range_km = BATTERY / pred_te * 100
range_abs_error    = np.abs(actual_range_km - predicted_range_km)

range_metrics = {
    '평균 절대오차': range_abs_error.mean(),
    '중앙값': np.median(range_abs_error),
    '90% 분위': np.quantile(range_abs_error, 0.90),
    '95% 분위': np.quantile(range_abs_error, 0.95),
    '최대': range_abs_error.max(),
}
for k, v in range_metrics.items():
    print(f'{k}: {v:.1f} km')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(range_abs_error, bins=50, color='#5B7C99', alpha=0.85, edgecolor='white')
ax.axvline(range_abs_error.mean(), color='#E07B39', lw=2,
           label=f'평균 {range_abs_error.mean():.1f}km')
ax.axvline(np.quantile(range_abs_error, 0.95), color='#C0392B', ls='--', lw=2,
           label=f'95% 분위 {np.quantile(range_abs_error, 0.95):.1f}km')
ax.set_xlabel('주행가능거리 절대오차 (km)'); ax.set_ylabel('빈도')
ax.set_title('60kWh 배터리 기준 주행가능거리 오차 분포')
ax.legend(); plt.tight_layout(); plt.show()

**해석** : 60kWh 기준 **평균 절대오차는 약 8.9km, 중앙값은 약 7.1km**다.
테스트 사례의 90%는 약 18.5km, 95%는 약 22.9km 이내이며 최대 오차는 약 57.3km다.
분포가 오른쪽으로 긴 꼬리를 갖는 이유는 저전비(장거리 주행 가능) 구간에서 같은 전비 오차가
더 큰 거리 오차로 증폭되기 때문이다.

**판단** : 평균 오차뿐 아니라 **극단 조건의 꼬리 오차**를 고려한 보수적 잔량 안내가 필요하다
(예: 95% 분위를 안전 마진으로 사용). 예측구간 제공은 7장의 향후 과제로 제안한다.

### 5-7. 고정 전비 기준선과의 비교 — 이 모델은 얼마나 나은가

현행 방식(모든 상황에 단일 전비 적용)을 `DummyRegressor`로 재현해 정량 비교한다.

In [ ]:
fixed_model = DummyRegressor(strategy='mean').fit(X_trval[FINAL_FEATS], y_trval)
fixed_pred  = fixed_model.predict(X_te[FINAL_FEATS])
fixed_range_error = np.abs(actual_range_km - BATTERY / fixed_pred * 100)

# 참고: MAE 기준 최적 상수는 중앙값이므로 함께 확인
median_const = np.full(len(y_te), y_trval.median())
print(f'상수 기준선 Test MAE — 평균 {mean_absolute_error(y_te, fixed_pred):.6f} / '
      f'중앙값 {mean_absolute_error(y_te, median_const):.6f} (차이 무시 가능)')

comparison = pd.DataFrame([
    {'방식': '고정 전비 기준선', 'R²': r2_score(y_te, fixed_pred),
     '전비 MAE': mean_absolute_error(y_te, fixed_pred),
     '전비 RMSE': np.sqrt(mean_squared_error(y_te, fixed_pred)),
     '평균 거리오차(km)': fixed_range_error.mean(),
     '거리오차 95%(km)': np.quantile(fixed_range_error, 0.95)},
    {'방식': '주행조건 예측모델', 'R²': r2, '전비 MAE': mae, '전비 RMSE': rmse,
     '평균 거리오차(km)': range_abs_error.mean(),
     '거리오차 95%(km)': np.quantile(range_abs_error, 0.95)},
]).round(3)
print()
print(comparison.to_string(index=False))

mae_red   = (mean_absolute_error(y_te, fixed_pred) - mae) / mean_absolute_error(y_te, fixed_pred) * 100
range_red = (fixed_range_error.mean() - range_abs_error.mean()) / fixed_range_error.mean() * 100
print(f'\n전비 MAE 감소율: {mae_red:.1f}%   평균 거리오차 감소율: {range_red:.1f}%')

**해석** : 고정 전비 기준선은 전비 MAE 2.90, 평균 주행가능거리 오차 31.3km, 95% 오차 77.1km를 보인다.
주행조건 예측모델은 이를 각각 0.80과 8.9km로 낮춰 **약 72% 개선**했다.

이 기준선은 카탈로그 값을 임의로 고른 것이 아니라 **학습 데이터 분포에 맞춰 보정한 상수**다
(MAE 기준 최적 상수인 중앙값과 평균의 차이는 무시할 수준으로 확인됨).
즉 상수 예측기로서는 강한 기준선이며, 그럼에도 72% 개선이 나왔으므로 비교 결과는 보수적이다.

---
## 6. 활용 — 경로 에너지 비용 함수와 모델 저장

### 6-1. 예측 전비 → 경로 안내

$$E_{\text{경로}} = \sum_i \hat{y}_i \times \frac{d_i}{100}, \qquad
\text{도착 SOC} = \text{현재 SOC} - \frac{E_{\text{경로}}}{\text{배터리 용량}} \times 100$$

$\hat{y}_i$는 링크 $i$의 예측 전비, $d_i$는 링크 거리(km)다. 거리는 예측 입력이 아니라 변환 계수로만 쓰인다.

In [ ]:
def predict_route(segments, vehicle_state, soc_now=80, battery_kwh=60):
    """링크 리스트 + 차량 상태 → (총 에너지, 도착 SOC, 링크별 예측 전비)"""
    rows = []
    for seg in segments:
        row = {**vehicle_state, **seg}
        row['temp_dev_sq'] = (row['ambient_temp_C'] - 20) ** 2
        rows.append(row)
    data = pd.DataFrame(rows)

    pred_ev = final_pipe.predict(data[FINAL_FEATS])          # 거리는 입력에 없음
    segment_kwh = pred_ev * data[DISTANCE_COL].to_numpy() / 100   # 거리는 여기서만 사용

    total_kwh = segment_kwh.sum()
    arrive_soc = soc_now - total_kwh / battery_kwh * 100
    return total_kwh, arrive_soc, pred_ev

vehicle = dict(payload_kg=300, ambient_temp_C=-5, hvac_power_kw=3.5,
               battery_temp_C=20, driving_style_index=0.5, tire_pressure_bar=2.4)

route_A = [dict(speed_kmh=115, road_grade_pct=1.0, trip_distance_km=50),
           dict(speed_kmh=120, road_grade_pct=0.0, trip_distance_km=50)]
route_B = [dict(speed_kmh=60,  road_grade_pct=1.0, trip_distance_km=53),
           dict(speed_kmh=65,  road_grade_pct=0.0, trip_distance_km=53)]

for name, route in [('경로 A (고속도로 100km · 평균 118km/h)', route_A),
                    ('경로 B (국도 106km · 평균 62km/h)', route_B)]:
    total, soc, _ = predict_route(route, vehicle)
    print(f'{name}: 예상 소비 {total:.1f} kWh | 도착 SOC {soc:.1f}%')

**해석** : 거리가 6km 더 긴 국도 경로가 총 에너지에서는 앞선다.
고속 주행의 공기저항 페널티가 거리 이득을 상쇄하기 때문이며, 모델이 이 트레이드오프를 정량 판별한다.
이것이 "최단거리·최단시간"에 **"최소 에너지"라는 세 번째 안내 축**을 추가하는 방식이다.

*설계 노트* : 출발지·목적지가 같으면 경로 간 순 고도차는 동일하므로, 선형 모델에서 경사 항의 총합은
경로를 차별화하지 못한다(오르막 소비와 내리막 회수가 대칭 처리됨). 경로 간 차이를 만드는 것은 속도이며,
회생제동의 비대칭 효율은 7장의 한계로 명시한다.

### 6-2. 분할 불변성 자동 검증

3-2에서 세운 요건이 실제로 충족되는지 확인한다. 동일한 물리 경로를 서로 다른 개수의 링크로
표현했을 때 총 에너지가 같아야 한다.

In [ ]:
def check_segment_consistency(total_distance=100):
    common = dict(speed_kmh=80, payload_kg=300, ambient_temp_C=10, hvac_power_kw=2.0,
                  road_grade_pct=1.0, battery_temp_C=25, driving_style_index=0.5,
                  tire_pressure_bar=2.4)
    link_keys = ['speed_kmh', 'road_grade_pct']
    vehicle_state = {k: v for k, v in common.items() if k not in link_keys}

    rows = []
    for n in [1, 2, 5, 10, 20]:
        d = total_distance / n
        route = [{**{k: common[k] for k in link_keys}, DISTANCE_COL: d} for _ in range(n)]
        total, _, _ = predict_route(route, vehicle_state)
        rows.append({'링크 수': n, '링크당 거리(km)': d, '총 에너지(kWh)': total})
    return pd.DataFrame(rows)

consistency = check_segment_consistency()
print(consistency.to_string(index=False))

assert np.allclose(consistency['총 에너지(kWh)'], consistency['총 에너지(kWh)'].iloc[0], atol=1e-10)
print('\n통과: 동일 경로는 링크 분할 방식과 무관하게 같은 에너지를 산출합니다.')

**해석** : 1개 링크든 20개 링크든 총 에너지가 동일하다(3-2의 거리 포함 모델은 4% 차이).
설계 결정이 의도한 효과를 그대로 달성했음을 자동 검증으로 확인했다.

### 6-3. 모델 저장 — 서비스(Streamlit/FastAPI)로 전달

앱이 **단일 진실 공급원(single source of truth)** 으로 사용할 수 있도록,
파이프라인과 함께 피처 목록·파생변수 정의·입력 범위·성능을 메타데이터로 저장한다.

In [ ]:
FEATURE_BOUNDS = {c: [float(df[c].min()), float(df[c].max())] for c in ORIG}
FEATURE_BOUNDS[DISTANCE_COL] = [0.1, float(df[DISTANCE_COL].max())]

artifact = {
    'artifact_version': '2.0.0',
    'model_name': 'ev-energy-linear-no-distance',
    'pipeline': final_pipe,
    'features': FINAL_FEATS,
    'raw_features': ORIG,
    'derived_features': {
        'temp_dev_sq': {'source': 'ambient_temp_C', 'formula': '(x - 20) ** 2'},
    },
    'distance_col': DISTANCE_COL,
    'distance_usage': '전비 모델 입력에서 제외. 총에너지 = 예측전비 × 거리/100 계산에만 사용',
    'feature_bounds': FEATURE_BOUNDS,
    'target': TARGET,
    'metrics': {
        'cv_r2': round(cv_r2.mean(), 4), 'cv_mae': round(cv_mae.mean(), 4),
        'test_r2': round(r2, 4), 'test_mae': round(mae, 4), 'test_rmse': round(rmse, 4),
        'range_mae_km': round(range_abs_error.mean(), 2),
        'range_p95_km': round(np.quantile(range_abs_error, 0.95), 2),
    },
    'sklearn_version': sklearn.__version__,
}

# 저장 전 자체 정합성 확인: 파이프라인 내부 피처와 메타데이터가 일치하는가
assert list(final_pipe.feature_names_in_) == FINAL_FEATS
joblib.dump(artifact, 'ev_energy_model.pkl')
print('저장 완료: ev_energy_model.pkl  (artifact_version 2.0.0)')
print('피처', len(FINAL_FEATS), '개:', FINAL_FEATS)

---
## 7. 결론 · 한계 · 향후 과제

### 결론

본 프로젝트는 양산차에서 확보 가능한 주행·환경 신호 8개와 온도 파생변수 1개를 활용해
주행조건별 전비를 예측했다. 거리는 전비 모델에서 제외하고 총에너지 산출 단계에서만 사용함으로써
**링크 분할 방식과 무관한 경로 비용 함수**를 구성했다.

- 거리는 분명한 예측력(r=0.17)을 가지고 있었지만, 전비 모델에 포함하면 동일 경로의 링크 분할 방식에 따라
  에너지 비용이 약 4% 달라졌다. **설명력 약 2.3%p를 포기하고 분할 불변성과 서비스 일관성을 선택**했다.
- 다수 구성을 비교(5개 계열 기본 + 전 계열 튜닝 + 파생변수 ablation + 2차 다항 전수 탐색)한 결과,
  데이터 구조가 저공선성·대체로 선형·온도 비선형 1곳임을 진단하고
  **`(t−20)²` 파생변수를 더한 선형 회귀**를 최종 선정했다. 다항 전수 탐색의 상한 성능을 9개 피처로 재현했다.
- 성능은 반복 교차검증 R² 0.931 ± 0.003, MAE 0.795이며 홀드아웃 Test에서도 동일 수준으로 확인됐다.
  60kWh 기준 주행가능거리 평균 절대오차는 8.9km(중앙값 7.1km, 95% 22.9km)다.
- 학습 분포에 맞춰 보정한 **강한 고정 전비 기준선 대비 전비·거리오차를 모두 약 72% 감소**시켰다.
- 회귀 계수는 경로 비용 함수의 가중치로 직접 사용 가능하다(경사 +1%p = +0.50, 혹한 -10°C = +3.2).

### 한계

1. **합성/교육용 데이터** : 실차 CAN 로그 대비 노이즈·결측·센서 드리프트가 없어 성능이 낙관적일 수 있다.
   또한 입력이 DOE형으로 균등 분포되어 실도로의 조건 분포와 다르다.
2. **Test의 독립성 제약** : 거리 변수 제외를 검토하는 과정에서 Test 성능을 관찰했으므로,
   Test는 완전한 외부 검증이 아니다. 반복 교차검증을 주 지표로 보고했다.
3. **분석 단위 갭** : 데이터의 주행 단위(8~200km)는 실제 내비 링크(수백 m)보다 길다.
   링크 단위 적용 시 재검증이 필요하다.
4. **미반영 요인** : 회생제동의 비대칭 효율(선형 모델은 오르막·내리막을 대칭 처리),
   차종·배터리 열화도, 바람·강수, 정체 시 가감속 패턴.
5. **인과 해석 아님** : 회귀 계수는 다른 입력이 동일할 때의 조건부 관계이며,
   실차에서의 개입 효과를 의미하지 않는다.

### 향후 과제

- 실차 주행 로그(OBD/CAN) 기반 재학습 및 **링크 단위 외부 검증**
- 차종별 계수 보정(transfer) 체계 — 단일 모델을 차종 파라미터로 미세 조정
- **도착 SOC 예측구간 제공** — 5-6의 꼬리 오차(95% 22.9km)를 안전 마진으로 반영한 보수적 안내
- 회생제동 비대칭을 반영한 경사 항 분리(오르막 계수 ≠ 내리막 계수)
